# DSM - Example 3

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)

UC with deferrable demand and DSM cost (`DSM_Cost = 9.2`) in objective.

In [1]:
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, Objective, Constraint, SolverFactory,
    Binary, NonNegativeReals, minimize, value
)

# ---- Data: loaded from external file 'DSM_IC_e3_data.txt' ----
import sys, pathlib
# Make the ampl_data parser importable (one level up from this notebook)
_pkg = pathlib.Path.cwd().parent
if str(_pkg) not in sys.path: sys.path.insert(0, str(_pkg))
from ampl_data import parse_ampl_data

_d = parse_ampl_data('DSM_IC_e3_data.txt')

GEN_data     = _d['GEN']
PERIOD_data  = _d['PERIOD']
gen_min      = _d['gen_min']
gen_max      = _d['gen_max']
gen_RRlimit  = _d['gen_RRlimit']
gen_OpCost   = _d['gen_OpCost']
gen_NlCost   = _d['gen_NlCost']
gen_SuCost   = _d['gen_SuCost']
Time_TotalPd = _d['Time_TotalPd']
DSM_d_Limit  = _d['DSM_d_Limit']

DSM_Cost = 9.2

BigM = 1e3

m = ConcreteModel()
m.GEN    = Set(initialize=GEN_data, ordered=True)
m.PERIOD = Set(initialize=PERIOD_data, ordered=True)

m.gen_min     = Param(m.GEN, initialize=gen_min)
m.gen_max     = Param(m.GEN, initialize=gen_max)
m.gen_RRlimit = Param(m.GEN, initialize=gen_RRlimit)
m.gen_OpCost  = Param(m.GEN, initialize=gen_OpCost)
m.gen_NlCost  = Param(m.GEN, initialize=gen_NlCost)
m.gen_SuCost  = Param(m.GEN, initialize=gen_SuCost)

m.Time_TotalPd = Param(m.PERIOD, initialize=Time_TotalPd)
m.DSM_d_Limit  = Param(m.PERIOD, initialize=DSM_d_Limit)

m.u  = Var(m.GEN, m.PERIOD, domain=Binary)
m.v  = Var(m.GEN, m.PERIOD, domain=Binary)
m.Pg = Var(m.GEN, m.PERIOD)
m.DSM_d = Var(m.PERIOD, domain=NonNegativeReals)

m.obj = Objective(
    rule=lambda mm: sum(mm.gen_OpCost[g]*mm.Pg[g,t]
                       + mm.gen_NlCost[g]*mm.u[g,t]
                       + mm.gen_SuCost[g]*mm.v[g,t]
                       for g in mm.GEN for t in mm.PERIOD)
       + sum(mm.DSM_d[t]*DSM_Cost for t in mm.PERIOD if t < mm.PERIOD.last()),
    sense=minimize
)

def pb_rule(mm,t):
    if t == mm.PERIOD.first():
        return sum(mm.Pg[g,t] for g in mm.GEN) == mm.Time_TotalPd[t] - mm.DSM_d[t]
    tp = mm.PERIOD.prev(t)
    return sum(mm.Pg[g,t] for g in mm.GEN) == mm.Time_TotalPd[t] - mm.DSM_d[t] + mm.DSM_d[tp]
m.PowerBalance = Constraint(m.PERIOD, rule=pb_rule)

m.genLimit_Min = Constraint(m.GEN, m.PERIOD, rule=lambda mm,g,t: mm.gen_min[g]*mm.u[g,t] <= mm.Pg[g,t])
m.genLimit_Max = Constraint(m.GEN, m.PERIOD, rule=lambda mm,g,t: mm.Pg[g,t] <= mm.gen_max[g]*mm.u[g,t])

def rr_up(mm,g,t):
    if t == mm.PERIOD.first(): return Constraint.Skip
    tp = mm.PERIOD.prev(t)
    return mm.Pg[g,t]-mm.Pg[g,tp] <= mm.gen_RRlimit[g]*mm.u[g,tp] + BigM*mm.v[g,t]
def rr_dn(mm,g,t):
    if t == mm.PERIOD.first(): return Constraint.Skip
    tp = mm.PERIOD.prev(t)
    return mm.Pg[g,tp]-mm.Pg[g,t] <= mm.gen_RRlimit[g]*mm.u[g,t] + BigM*(mm.v[g,t]-mm.u[g,t]+mm.u[g,tp])
m.genRRLimit_Up = Constraint(m.GEN, m.PERIOD, rule=rr_up)
m.genRRLimit_Dn = Constraint(m.GEN, m.PERIOD, rule=rr_dn)

def vu_rule(mm,g,t):
    if t == mm.PERIOD.first():
        return mm.v[g,t] >= mm.u[g,t]
    return mm.v[g,t] >= mm.u[g,t] - mm.u[g, mm.PERIOD.prev(t)]
m.genVU = Constraint(m.GEN, m.PERIOD, rule=vu_rule)

m.DSM_limit = Constraint(m.PERIOD, rule=lambda mm,t: mm.DSM_d[t] <= mm.DSM_d_Limit[t])
m.DSM_Zero  = Constraint(expr=m.DSM_d[m.PERIOD.last()] == 0)

model = m

In [2]:
# ---- Solve with Gurobi ----
solver = SolverFactory('gurobi')
solver.options['MIPGap'] = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)
m = model
print("g  t   v       u    Pg")
for g in m.GEN:
    for t in m.PERIOD:
        print(f"{g}  {t}   {value(m.v[g,t]):.2f}   {int(round(value(m.u[g,t])))}   {value(m.Pg[g,t]):.3f}")
if hasattr(m, "DSM_d"):
    print("\nDSM deferred load:")
    for t in m.PERIOD:
        print(f"  t={t}  DSM_d = {value(m.DSM_d[t]):.3f}")

Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmpty78mm55.pyomo.lp


Reading time = 0.00 seconds
x1: 45 rows, 32 columns, 126 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 20 logical processors, using up to 20 threads



Non-default parameters:
TimeLimit  90
MIPGap  0



Optimize a model with 45 rows, 32 columns and 126 nonzeros


Model fingerprint: 0xe8f0a430


...

Explored 1 nodes (19 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 20 (of 20 available processors)



Solution count 4: 5042 5044 5252 5260 



Optimal solution found (tolerance 0.00e+00)


Best objective 5.042000000000e+03, best bound 5.042000000000e+03, gap 0.0000%


ok optimal
g  t   v       u    Pg
1  1   -0.00   0   0.000
1  2   -0.00   0   0.000
2  1   -0.00   0   0.000
2  2   -0.00   0   0.000
3  1   1.00   1   50.000
3  2   -0.00   1   50.000
4  1   1.00   1   30.000
4  2   -0.00   0   0.000
5  1   1.00   1   40.000
5  2   -0.00   1   40.000

DSM deferred load:
  t=1  DSM_d = 10.000
  t=2  DSM_d = 0.000
